# Colab GPU runtime for Phillips_UC2

Run this notebook **in Google Colab** (Runtime > Change runtime type > GPU). It sets up an SSH tunnel so you can attach VS Code's **Remote - SSH** extension directly to this Colab VM and run/debug the repo on its GPU as if it were local.

Steps: run all cells below in order, then follow the printed VS Code instructions.

In [1]:
!nvidia-smi

Wed Jul 29 05:01:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# One-time setup: SSH server + cloudflared tunnel (no ngrok account needed, no flaky colab_ssh package)
!apt-get -qq -y install openssh-server > /dev/null
!mkdir -p /var/run/sshd
!echo "root:changeme" | chpasswd
!sed -i "s/#PermitRootLogin.*/PermitRootLogin yes/" /etc/ssh/sshd_config
!sed -i "s/#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config
!service ssh restart

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared


 * Restarting OpenBSD Secure Shell server sshd
   ...done.


In [3]:
import subprocess, time, re

# start tunnel in background, log to file so we can grab the printed URL
proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "ssh://localhost:22", "--logfile", "/content/cloudflared.log"]
)

url = None
for _ in range(30):
    time.sleep(1)
    try:
        log = open("/content/cloudflared.log").read()
    except FileNotFoundError:
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+trycloudflare\.com", log)
    if match:
        url = match.group(0)
        break

if not url:
    raise RuntimeError("cloudflared didn't print a URL in time -- check /content/cloudflared.log")

hostname = url.replace("https://", "")
print(f"""Add this to your local ~/.ssh/config, then VS Code Remote-SSH -> Connect to Host -> colab
(password: changeme)

Host colab
    HostName {hostname}
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h
""")


Add this to your local ~/.ssh/config, then VS Code Remote-SSH -> Connect to Host -> colab
(password: changeme)

Host colab
    HostName someone-dancing-improving-wireless.trycloudflare.com
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h



The cell above prints an SSH `Host` block. It requires `cloudflared` installed **locally** too (the `ProxyCommand` runs it on your machine) — install it from https://github.com/cloudflare/cloudflared/releases or `winget install --id Cloudflare.cloudflared`.

Paste the block into your local `~/.ssh/config` (VS Code Command Palette > "Remote-SSH: Open SSH Configuration File"), save, then Command Palette > "Remote-SSH: Connect to Host" > `colab`. Enter password `changeme` when prompted.

In [ ]:
# Get the repo onto the Colab VM
!git clone https://github.com/sormazabal/Phillips_UC2.git /content/Phillips_UC2


fatal: destination path '/content/Phillips_UC2' already exists and is not an empty directory.


In [ ]:
%cd /content/Phillips_UC2
!pip install -q -r requirements.txt

/content/Phillips_UC2


In [ ]:
# Mount data from colab
from google.colab import drive
drive.mount('/content/drive')

# ponytail: shared folders only show up under MyDrive if you added a shortcut to them first
# (open the Drive link -> right-click the folder -> "Add shortcut to Drive"), then set the name below.
DRIVE_DATASET_PATH = '/content/drive/MyDrive/Arcade'
!ln -sfn "$DRIVE_DATASET_PATH" /content/Phillips_UC2/Arcade
!ls /content/Phillips_UC2/Arcade


Mounted at /content/drive
stenosis  syntax


In [ ]:
#load checkpoint from the drive
import glob, os, shutil

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/Arcade/checkpoints'  # ponytail: adjust if checkpoints live elsewhere on Drive
os.makedirs('checkpoints', exist_ok=True)

ckpts = sorted(glob.glob(f'{DRIVE_CHECKPOINT_DIR}/*.ckpt'), key=os.path.getmtime)
if not ckpts:
    raise FileNotFoundError(f"No .ckpt files found in {DRIVE_CHECKPOINT_DIR}")

latest_ckpt = ckpts[-1]
local_ckpt = os.path.join('checkpoints', os.path.basename(latest_ckpt))
shutil.copy(latest_ckpt, local_ckpt)
print(f"Loaded latest checkpoint: {latest_ckpt} -> {local_ckpt}")


In [ ]:
# Load secrets from the .env file in Drive (HF_TOKEN, WANDB_API_KEY, etc.)
import os

env_path = '/content/drive/MyDrive/Env_vars/.env'
loaded = []
with open(env_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, _, value = line.partition('=')
        key = key.strip()
        os.environ[key] = value.strip().strip('"').strip("'")
        loaded.append(key)

print(f"Loaded from {env_path}: {loaded}")


Loaded from /content/drive/MyDrive/Env_vars/.env: ['GITHUB_TOKEN', 'HF_TOKEN']


From the VS Code window connected via Remote-SSH, open `/content/Phillips_UC2` as the workspace folder, select the Python interpreter, and run e.g.:

```bash
python scripts/train.py --config config.yaml
```

Keep this Colab tab open -- closing it kills the tunnel and the VM.

In [7]:
%env PYTHONPATH=/content/Phillips_UC2
!python scripts/train.py --config config.yaml

env: PYTHONPATH=/content/Phillips_UC2
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
config.json: 100% 70.0k/70.0k [00:00<00:00, 51.2MB/s]

pytorch_model.bin: downloading bytes:  87% 284M/328M [00:02<00:00, 134MB/s, 24.3MB/s  ]  
pytorch_model.bin: downloading bytes:  92% 302M/328M [00:02<00:00, 120MB/s, 25.5MB/s  ]
pytorch_model.bin: reconstructing file:  82% 268M/328M [00:03<00:00, 85.8MB/s, 6.25MB/s  ]
pytorch_model.bin: downloading bytes: 100% 310M/310M [00:03<00:00, 79.0MB/s, 26.2MB/s  ]] 
pytorch_model.bin: reconstructing file: 100% 328M/328M [00:03<00:00,

In [ ]:
# Option 1: run as a module from repo root (preferred, no code changes)
cd /content/Phillips_UC2
python -m scripts.train --config config.yaml
